# Facial expression synchrony per month (zero lag)


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from notebook_setup import init
FINAL_DIR, PDF_DIR = init()
import plot_style

from analysis_config import BASE_PATH, FOCUS_AUS
from dyad_analysis_utils import (
    load_openface_data,
    load_prepared_meetings,
    organize_sessions_by_period,
    compute_averaged_correlation_matrix,
    compute_pvalue_matrix,
    holm_correct_matrix,
    plot_heatmaps,
    plot_pvalue_heatmaps,
)

base_path = BASE_PATH
if not base_path.exists():
    raise FileNotFoundError(f"Expected directory not found: {base_path.resolve()}")

meeting_groups, input_paths = load_openface_data(base_path)
print(f"Found {len(input_paths)} session folders, {len(meeting_groups)} meetings")
print(f"PDF export dir: {PDF_DIR}")

In [ ]:
df_infant_arr, df_adult_arr, AUs = load_prepared_meetings(meeting_groups)
print(f"Loaded {len(df_infant_arr)} meetings, {len(AUs)} AU columns")


In [ ]:
period_aggregated_data, all_periods_data = organize_sessions_by_period(
    df_infant_arr, df_adult_arr, AUs
)
for p in (1, 2, 3):
    print(f"Period {p}: {len(period_aggregated_data[p])} sessions")
print(f"Total: {len(all_periods_data)} sessions")


In [ ]:
print(f"Using FOCUS_AUS ({len(FOCUS_AUS)}): {FOCUS_AUS}")


In [ ]:
period_corr_matrices = {}
period_pvalue_matrices = {}
for period in (1, 2, 3):
    period_corr_matrices[period] = compute_averaged_correlation_matrix(
        period_aggregated_data[period], FOCUS_AUS
    )
    period_pvalue_matrices[period] = compute_pvalue_matrix(
        period_aggregated_data[period], FOCUS_AUS
    )
    print(f"Period {period}: computed from {len(period_aggregated_data[period])} sessions")


In [ ]:
for period in (1, 2, 3):
    mean_abs_r = period_corr_matrices[period].abs().values.mean()
    print(f"{plot_style.month_label(period)}: mean |r| = {mean_abs_r:.3f}")


In [ ]:
plot_heatmaps(
    period_corr_matrices,
    FOCUS_AUS,
    title="Direct correlation: infant vs adult AU",
    vmin=-0.3,
    vmax=0.3,
    cmap="coolwarm",
    xlabel="Adult AU",
    ylabel="Infant AU",
    cbar_label="Pearson r",
    figsize=(10.5, 3.6),
    annotate=True,
    pvalue_matrices=period_pvalue_matrices,
    save_pdf_path=PDF_DIR / "direct_corr_heatmaps.pdf",
)


In [ ]:
for period in [1, 2, 3]:
    n_sig = (period_pvalue_matrices[period].values <= 0.05).sum()
    print(f"{plot_style.month_label(period)}: {n_sig} AU pairs with p <= .05")

plot_pvalue_heatmaps(
    period_pvalue_matrices,
    FOCUS_AUS,
    title="Direct correlation p-values: infant vs adult AU",
    alpha=0.05,
    xlabel="Adult AU",
    ylabel="Infant AU",
    figsize=(10.5, 3.6),
    save_pdf_path=PDF_DIR / "direct_corr_pvalues_heatmaps.pdf",
)


In [ ]:
period_pvalue_matrices_holm = {
    period: holm_correct_matrix(period_pvalue_matrices[period], alpha=0.05)
    for period in (1, 2, 3)
}
for period in (1, 2, 3):
    p_raw = period_pvalue_matrices[period]
    p_holm = period_pvalue_matrices_holm[period]
    n_tests = int(p_raw.notna().values.sum())
    n_raw = int((p_raw.values <= 0.05).sum())
    n_holm = int((p_holm.values <= 0.05).sum())
    print(f"{plot_style.month_label(period)}: {n_raw}/{n_tests} uncorrected, {n_holm}/{n_tests} after Holm")
